# 🏥 Medical Visual Question Answering (Med-VQA) & Radiology Report Generation

> **Runtime:** Google Colab · T4 GPU (16 GB VRAM) · Python 3.10

---

## 📋 Tutorial Overview

This notebook walks end-to-end through building a **Medical AI pipeline** that:

| Stage | What happens |
|---|---|
| **1. Setup** | Install deps, verify GPU, authenticate HuggingFace |
| **2. Dataset** | Load & explore VQA-RAD (radiology VQA pairs + images) |
| **3. Baseline** | Zero-shot Med-VQA with BLIP-2 (OPT-2.7B, 4-bit) |
| **4. PEFT/LoRA** | Fine-tune the LM head with LoRA — fits in T4 VRAM |
| **5. Training** | `Trainer` supervised fine-tuning loop |
| **6. Evaluation** | BLEU, token-F1, qualitative comparison |
| **7. Pipeline** | Multi-question VQA → structured diagnostic report |
| **8. Demo** | Interactive single-image inference widget |

### 🎯 Learning Objectives
- Understand the **Med-VQA task** formulation and challenges  
- Apply **4-bit quantisation + LoRA** to run large vision-language models on consumer GPUs  
- Build a **report generation pipeline** from raw X-ray images  
- Evaluate medical NLP outputs with appropriate metrics

> ⚠️ **Disclaimer:** This notebook is for *educational purposes only*. Outputs must **never** be used for real clinical diagnosis.

---
## Section 1 · Environment Setup & GPU Verification

In [ ]:
# ─── Install all dependencies ────────────────────────────────────────────────
# Run once; restart kernel if prompted
!pip install -q \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.29.0 \
    trl>=0.8.6 \
    datasets>=2.18.0 \
    evaluate>=0.4.1 \
    rouge_score \
    nltk \
    Pillow \
    matplotlib \
    seaborn \
    scikit-learn \
    huggingface_hub

print("✅ Packages installed.")

In [ ]:
import subprocess, sys, os
import torch

# ─── GPU diagnostic ──────────────────────────────────────────────────────────
print("=" * 55)
print(f"Python        : {sys.version.split()[0]}")
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram_gb = gpu.total_memory / 1e9
    print(f"GPU           : {gpu.name}")
    print(f"VRAM          : {vram_gb:.1f} GB")
    if vram_gb < 10:
        print("⚠️  Less than 10 GB VRAM — use 4-bit quantisation (enabled below).")
    else:
        print("✅ VRAM sufficient for 4-bit BLIP-2 + LoRA fine-tuning.")
else:
    print("❌ No GPU. Go to Runtime → Change runtime type → T4 GPU.")
print("=" * 55)

In [ ]:
import random, numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

---
## Section 2 · Dataset: VQA-RAD (Radiology Visual QA)

**VQA-RAD** contains ~315 radiology images paired with 3,515 clinician-authored Q&A pairs.

| Attribute | Detail |
|---|---|
| Source | `flaviagiammarino/vqa-rad` on HuggingFace Hub |
| Modalities | Chest X-ray, Head CT, Abdomen |
| Answer types | Closed (yes/no) & Open-ended |
| Split | 2,248 train / 451 test QA pairs |

In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import textwrap, warnings, collections
import seaborn as sns
warnings.filterwarnings("ignore")

print("Loading VQA-RAD dataset...")
dataset = load_dataset("flaviagiammarino/vqa-rad")
train_ds = dataset["train"]
test_ds  = dataset["test"]
print(dataset)
print(f"\nTrain: {len(train_ds)}  |  Test: {len(test_ds)}")
print(f"Features: {list(train_ds.features.keys())}")

In [ ]:
# ─── Visualise 6 diverse samples ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("VQA-RAD — Sample Images with Q&A Pairs",
             fontsize=14, fontweight="bold", y=1.01)
for ax, idx in zip(axes.flat, [0, 50, 120, 200, 350, 500]):
    s = train_ds[idx]
    ax.imshow(s["image"].convert("RGB"), cmap="gray")
    ax.axis("off")
    q = textwrap.fill(f"Q: {s['question']}", 35)
    ax.set_title(f"{q}\nA: {s['answer']}", fontsize=8, loc="left", pad=4)
plt.tight_layout(); plt.show()

In [ ]:
# ─── Answer-type distribution & top answers ───────────────────────────────────
ans_types   = collections.Counter(train_ds["answer_type"])
top_answers = collections.Counter(train_ds["answer"]).most_common(20)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.pie(ans_types.values(), labels=ans_types.keys(),
        autopct="%1.1f%%", startangle=140,
        colors=sns.color_palette("Set2"))
ax1.set_title("Answer Type Distribution", fontweight="bold")

labels, counts = zip(*top_answers)
ax2.barh(labels[::-1], counts[::-1],
         color=sns.color_palette("Blues_d", len(labels)))
ax2.set_xlabel("Count")
ax2.set_title("Top 20 Most Frequent Answers", fontweight="bold")
plt.tight_layout(); plt.show()
print(f"Unique questions: {len(set(train_ds['question']))}")
print(f"Unique answers  : {len(set(train_ds['answer']))}")

In [ ]:
# ─── Preprocessing utilities ─────────────────────────────────────────────────
def preprocess_image(image: Image.Image, size: int = 224) -> Image.Image:
    """Resize to square and ensure RGB mode."""
    return image.convert("RGB").resize((size, size), Image.LANCZOS)

def format_prompt(question: str, context: str = "") -> str:
    """Format a Med-VQA prompt for the language model."""
    ctx = f" Context: {context}." if context else ""
    return (
        f"You are a radiologist assistant analysing a medical image.{ctx}\n"
        f"Question: {question}\n"
        f"Answer:"
    )

print("Sample prompt:")
print(format_prompt(train_ds[0]["question"]))

---
## Section 3 · Baseline Med-VQA with BLIP-2 (Zero-Shot)

**BLIP-2** connects a frozen vision encoder (EVA-CLIP) to a frozen LLM (OPT-2.7B)
via a lightweight **Q-Former** bridge. Loaded in **4-bit NF4 quantisation** to fit T4 VRAM.

```
X-Ray ──► EVA-CLIP Encoder ──► Q-Former ──► OPT-2.7B ──► Text Answer
           (frozen)            (trainable)   (frozen)
```

In [ ]:
from transformers import (
    Blip2Processor,
    Blip2ForConditionalGeneration,
    BitsAndBytesConfig,
)

MODEL_ID = "Salesforce/blip2-opt-2.7b"

# ─── 4-bit NF4 quantisation ───────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",             # NormalFloat4
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,        # nested quantisation
)

print(f"Loading '{MODEL_ID}' with 4-bit quantisation...")
print("This may take 2-3 min on first run (downloads ~5 GB).")

processor = Blip2Processor.from_pretrained(MODEL_ID)
model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()

if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n✅ Loaded. VRAM: {used:.1f} / {total:.1f} GB")

total_p    = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_p/1e6:.1f}M")
print(f"Trainable params: {trainable_p/1e6:.1f}M")

In [ ]:
@torch.inference_mode()
def predict_answer(image, question, max_new_tokens=64, num_beams=4):
    """Run BLIP-2 zero-shot VQA on a single image-question pair."""
    prompt = format_prompt(question)
    inputs = processor(
        images=image, text=prompt, return_tensors="pt"
    ).to(DEVICE, torch.float16)
    gen_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        num_beams=num_beams, repetition_penalty=1.3,
    )
    answer = processor.tokenizer.batch_decode(
        gen_ids, skip_special_tokens=True
    )[0].strip()
    if "Answer:" in answer:
        answer = answer.split("Answer:")[-1].strip()
    return answer

# ─── Spot-check 5 samples ─────────────────────────────────────────────────────
print("Zero-shot predictions on 5 test samples")
print("=" * 65)
for i in range(5):
    s   = test_ds[i]
    img = preprocess_image(s["image"])
    pred = predict_answer(img, s["question"])
    print(f"[{i+1}] Q : {s['question']}")
    print(f"    GT: {s['answer']}")
    print(f"    PR: {pred}")
    print("-" * 65)

In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

def compute_token_f1(pred: str, ref: str) -> float:
    pred_tokens = pred.lower().split()
    ref_tokens  = ref.lower().split()
    common   = collections.Counter(pred_tokens) & collections.Counter(ref_tokens)
    n_common = sum(common.values())
    if n_common == 0:
        return 0.0
    precision = n_common / len(pred_tokens)
    recall    = n_common / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

EVAL_N   = 100
smoother = SmoothingFunction().method1
bleu_scores, f1_scores, exact_matches = [], [], []

print(f"Evaluating baseline on {EVAL_N} test samples...")
for i in range(EVAL_N):
    s    = test_ds[i]
    img  = preprocess_image(s["image"])
    pred = predict_answer(img, s["question"], max_new_tokens=32)
    ref  = s["answer"]
    r_tok = nltk.word_tokenize(ref.lower())
    p_tok = nltk.word_tokenize(pred.lower())
    bleu_scores.append(sentence_bleu([r_tok], p_tok, smoothing_function=smoother))
    f1_scores.append(compute_token_f1(pred, ref))
    exact_matches.append(int(pred.strip().lower() == ref.strip().lower()))

print("\n" + "=" * 40)
print(f"  BLEU-1 (avg)  : {np.mean(bleu_scores):.4f}")
print(f"  Token F1 (avg): {np.mean(f1_scores):.4f}")
print(f"  Exact Match   : {np.mean(exact_matches)*100:.1f}%")
print("=" * 40)

---
## Section 4 · PEFT/LoRA Configuration

### Why LoRA for Medical Imaging?

| Full Fine-tuning | LoRA (Low-Rank Adaptation) |
|---|---|
| Updates all ~2.7B params | Updates only ~1–4M params |
| Requires 40+ GB VRAM | Fits in 16 GB T4 |
| Risk of catastrophic forgetting | Preserves pre-trained knowledge |

### LoRA Mathematics

For weight matrix **W** ∈ ℝ^(d×k):

```
W' = W + ΔW = W + B · A
```

**B** ∈ ℝ^(d×r), **A** ∈ ℝ^(r×k), rank **r ≪ min(d, k)**.  
Only A and B are trained; W is frozen.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"]

lora_config = LoraConfig(
    r=16,                     # LoRA rank
    lora_alpha=32,            # scaling factor
    target_modules=TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

print("LoRA configuration:")
print(f"  rank (r)       : {lora_config.r}")
print(f"  alpha          : {lora_config.lora_alpha}")
print(f"  dropout        : {lora_config.lora_dropout}")
print(f"  target modules : {TARGET_MODULES}")
print(f"  effective scale: {lora_config.lora_alpha / lora_config.r:.2f}")

In [ ]:
# ─── Prepare model & inject LoRA adapters ────────────────────────────────────
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

total    = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nFrozen : {(total-trainable)/1e6:.1f}M  (vision encoder + base LM)")
print(f"LoRA   : {trainable/1e6:.2f}M  (A & B matrices only)")

In [ ]:
# ─── Visualise LoRA param distribution ───────────────────────────────────────
lora_layers = [
    (name, p.numel())
    for name, p in model.named_parameters()
    if p.requires_grad and "lora" in name.lower()
]
module_summary = collections.defaultdict(int)
for name, cnt in lora_layers:
    key = name.split(".")[-2] if "." in name else name
    module_summary[key] += cnt

fig, ax = plt.subplots(figsize=(9, 4))
keys = list(module_summary.keys())
vals = [module_summary[k] / 1e3 for k in keys]
bars = ax.barh(keys, vals, color=sns.color_palette("viridis", len(keys)))
ax.bar_label(bars, fmt="%.1f K", padding=4)
ax.set_xlabel("Trainable Parameters (×1000)")
ax.set_title("LoRA Trainable Parameters per Module Type", fontweight="bold")
ax.set_xlim(0, max(vals) * 1.25)
plt.tight_layout(); plt.show()
print(f"Total LoRA layers injected: {len(lora_layers)}")

---
## Section 5 · Fine-tuning with Hugging Face Trainer

Training setup:
- **Gradient accumulation** → effective batch size of 16
- **fp16 mixed precision** → halves memory usage
- **Gradient checkpointing** → trades compute for memory
- **paged_adamw_8bit** → 8-bit AdamW saves ~4 GB VRAM
- **Cosine LR** with warmup → smooth convergence

In [ ]:
from torch.utils.data import Dataset as TorchDataset

class MedVQADataset(TorchDataset):
    """
    Tokenises image-question-answer triplets into BLIP-2 inputs.
    Labels mask the prompt so loss is computed only on the answer.
    """
    def __init__(self, hf_dataset, processor, max_length=128):
        self.data      = hf_dataset
        self.processor = processor
        self.max_length = max_length

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        s       = self.data[idx]
        image   = preprocess_image(s["image"])
        question, answer = s["question"], s["answer"]
        full_text = format_prompt(question) + " " + answer

        enc = self.processor(
            images=image, text=full_text,
            return_tensors="pt",
            padding="max_length", max_length=self.max_length, truncation=True,
        )
        input_ids      = enc["input_ids"].squeeze()
        attention_mask = enc["attention_mask"].squeeze()
        pixel_values   = enc["pixel_values"].squeeze()

        labels = input_ids.clone()
        prompt_len = self.processor.tokenizer(
            format_prompt(question), return_tensors="pt", padding=False,
        )["input_ids"].shape[-1]
        labels[:prompt_len] = -100   # mask prompt tokens from loss

        return {
            "input_ids": input_ids, "attention_mask": attention_mask,
            "pixel_values": pixel_values, "labels": labels,
        }

train_dataset = MedVQADataset(train_ds, processor, max_length=128)
eval_dataset  = MedVQADataset(test_ds,  processor, max_length=128)
print(f"Train: {len(train_dataset)}  |  Eval: {len(eval_dataset)}")

# Shape check
for k, v in train_dataset[0].items():
    print(f"  {k:20s}: {v.shape}")

In [ ]:
from transformers import TrainingArguments, Trainer

OUTPUT_DIR = "/content/medvqa_lora_checkpoint"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,       # effective BS = 16
    per_device_eval_batch_size=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    optim="paged_adamw_8bit",            # 8-bit AdamW
    fp16=True,
    gradient_checkpointing=True,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
    logging_dir=os.path.join(OUTPUT_DIR, "logs"),
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=SEED,
)
print("Training args ready.")
print(f"  Effective batch size : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate        : {training_args.learning_rate}")

In [ ]:
def medvqa_collator(features):
    """Stack list of dataset items into a batched tensor dict."""
    return {key: torch.stack([f[key] for f in features]) for key in features[0]}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=medvqa_collator,
)

print("Starting LoRA fine-tuning...")
print(f"Approx steps: ~{len(train_dataset)//(training_args.per_device_train_batch_size*training_args.gradient_accumulation_steps)*int(training_args.num_train_epochs)}")
train_result = trainer.train()
print(f"\n✅ Done! Train loss: {train_result.training_loss:.4f}  Runtime: {train_result.metrics['train_runtime']:.0f}s")

In [ ]:
# ─── Loss curves ─────────────────────────────────────────────────────────────
history      = trainer.state.log_history
train_losses = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_losses  = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

fig, ax = plt.subplots(figsize=(10, 4))
if train_losses:
    steps, losses = zip(*train_losses)
    ax.plot(steps, losses, label="Train loss", color="royalblue", linewidth=1.5)
if eval_losses:
    steps, losses = zip(*eval_losses)
    ax.plot(steps, losses, label="Eval loss", color="tomato",
            marker="o", linewidth=2, markersize=6)
ax.set_xlabel("Step"); ax.set_ylabel("Loss")
ax.set_title("LoRA Fine-tuning Loss Curves", fontweight="bold")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ─── Save LoRA adapter weights only (~20–40 MB vs ~5 GB full model) ──────────
ADAPTER_PATH = "/content/medvqa_lora_adapter"
model.save_pretrained(ADAPTER_PATH)
processor.save_pretrained(ADAPTER_PATH)

adapter_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, files in os.walk(ADAPTER_PATH) for f in files
) / 1e6
print(f"✅ Adapter saved to: {ADAPTER_PATH}")
print(f"   Size: {adapter_size:.1f} MB  (vs ~5,000 MB for full model)")

---
## Section 6 · Evaluation: Baseline vs. Fine-tuned

| Metric | What it measures |
|---|---|
| **BLEU-1** | n-gram overlap with reference answer |
| **Token F1** | Recall + precision at token level |
| **Exact Match** | Perfect string match (useful for closed QA) |
| **ROUGE-L** | Longest common subsequence overlap |

In [ ]:
# ─── Merge LoRA into base weights for faster inference ────────────────────────
try:
    tuned_model = model.merge_and_unload()
    tuned_model.eval()
    print("✅ LoRA adapters merged into base model.")
except Exception as e:
    print(f"Merge skipped ({e}); using PEFT model directly.")
    tuned_model = model
    tuned_model.eval()

In [ ]:
import evaluate as hf_evaluate
rouge = hf_evaluate.load("rouge")

@torch.inference_mode()
def predict_with_model(mdl, image, question, max_new_tokens=64):
    prompt = format_prompt(question)
    inputs = processor(
        images=image, text=prompt, return_tensors="pt"
    ).to(DEVICE, torch.float16)
    ids = mdl.generate(**inputs, max_new_tokens=max_new_tokens,
                        num_beams=4, repetition_penalty=1.3)
    out = processor.tokenizer.batch_decode(ids, skip_special_tokens=True)[0]
    if "Answer:" in out:
        out = out.split("Answer:")[-1].strip()
    return out

def evaluate_predictions(preds, refs):
    bleu_vals, f1_vals, em_vals = [], [], []
    smoother = SmoothingFunction().method1
    for p, r in zip(preds, refs):
        r_tok = nltk.word_tokenize(r.lower())
        p_tok = nltk.word_tokenize(p.lower())
        bleu_vals.append(sentence_bleu([r_tok], p_tok, smoothing_function=smoother))
        f1_vals.append(compute_token_f1(p, r))
        em_vals.append(int(p.strip().lower() == r.strip().lower()))
    rs = rouge.compute(predictions=preds, references=refs)
    return {
        "BLEU-1"      : np.mean(bleu_vals),
        "Token F1"    : np.mean(f1_vals),
        "Exact Match" : np.mean(em_vals) * 100,
        "ROUGE-L"     : rs["rougeL"],
    }

EVAL_N = 80
ft_preds  = []
gt_labels = []
print(f"Evaluating fine-tuned model on {EVAL_N} test samples...")
for i in range(EVAL_N):
    s = test_ds[i]
    img = preprocess_image(s["image"])
    ft_preds.append(predict_with_model(tuned_model, img, s["question"]))
    gt_labels.append(s["answer"])

ft_metrics   = evaluate_predictions(ft_preds, gt_labels)
base_metrics = {
    "BLEU-1"     : np.mean(bleu_scores[:EVAL_N]),
    "Token F1"   : np.mean(f1_scores[:EVAL_N]),
    "Exact Match": np.mean(exact_matches[:EVAL_N]) * 100,
    "ROUGE-L"    : 0.0,
}

print("\n" + "=" * 52)
print(f"{'Metric':<15} {'Baseline':>12} {'Fine-tuned':>12} {'Δ':>8}")
print("-" * 52)
for m in ["BLEU-1", "Token F1", "ROUGE-L"]:
    b, f = base_metrics[m], ft_metrics[m]
    sign = "+" if f >= b else ""
    print(f"{m:<15} {b:>12.4f} {f:>12.4f} {sign}{f-b:>7.4f}")
b, f = base_metrics["Exact Match"], ft_metrics["Exact Match"]
print(f"{'Exact Match':<15} {b:>11.1f}% {f:>11.1f}% {f-b:>+7.1f}%")
print("=" * 52)

In [ ]:
# ─── Side-by-side bar chart ───────────────────────────────────────────────────
metrics_to_plot = ["BLEU-1", "Token F1", "ROUGE-L"]
x, width = np.arange(len(metrics_to_plot)), 0.35
fig, ax = plt.subplots(figsize=(9, 5))
base_vals = [base_metrics[m] for m in metrics_to_plot]
ft_vals   = [ft_metrics[m]   for m in metrics_to_plot]
b1 = ax.bar(x - width/2, base_vals, width, label="Zero-Shot Baseline", color="steelblue",   alpha=0.85)
b2 = ax.bar(x + width/2, ft_vals,   width, label="LoRA Fine-tuned",   color="darkorange", alpha=0.85)
ax.bar_label(b1, fmt="%.3f", padding=3, fontsize=9)
ax.bar_label(b2, fmt="%.3f", padding=3, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(metrics_to_plot)
ax.set_ylabel("Score")
ax.set_ylim(0, max(max(base_vals), max(ft_vals)) * 1.3)
ax.set_title("Med-VQA: Zero-Shot vs LoRA Fine-tuned BLIP-2", fontweight="bold")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

---
## Section 7 · Report Generation Pipeline

A single QA pair is limited. A *diagnostic report* asks multiple targeted questions
and assembles them into a structured clinical note:

```
                 ┌──────────────────────────────────────────┐
 X-Ray / CT     │   Multi-question VQA Engine (BLIP-2)     │
      ──────►   │  Q1: Modality?       → "Chest X-ray"     │  ──► Structured
                │  Q2: Abnormalities?  → "Cardiomegaly"    │      Report
                │  Q3: Severity?       → "Moderate"        │
                │  Q4: Impression?     → "Heart failure"   │
                └──────────────────────────────────────────┘
```

In [ ]:
REPORT_QUESTIONS = {
    "modality"       : {"question": "What imaging modality is shown?",                            "section": "Study Type"},
    "view"           : {"question": "What is the radiographic view or orientation?",              "section": "View"},
    "organ_focus"    : {"question": "Which organ or body region is the primary focus?",           "section": "Region of Interest"},
    "abnormality"    : {"question": "Are there any visible abnormalities or pathological findings?", "section": "Abnormalities"},
    "specific_finding": {"question": "Describe specific findings: masses, effusions, infiltrates, consolidations.", "section": "Specific Findings"},
    "affected_side"  : {"question": "Is the finding unilateral or bilateral? Which side?",        "section": "Laterality"},
    "severity"       : {"question": "How severe are the findings: mild, moderate, or severe?",    "section": "Severity"},
    "impression"     : {"question": "What is the most likely clinical impression or diagnosis?",   "section": "Impression / Diagnosis"},
    "recommendation" : {"question": "What follow-up or clinical action would be recommended?",    "section": "Recommendation"},
}
print(f"Question bank: {len(REPORT_QUESTIONS)} targeted clinical questions.")

In [ ]:
import datetime

@torch.inference_mode()
def generate_report(image, model, processor,
                    questions=REPORT_QUESTIONS,
                    max_new_tokens=80,
                    patient_id="ANON-0001",
                    verbose=True):
    """
    Run multi-question VQA and assemble a structured diagnostic report.
    Returns dict with: patient_id, date, sections, raw_answers, formatted_text.
    """
    img = preprocess_image(image)
    sections, raw = [], {}

    if verbose:
        print(f"Generating report for {patient_id} ...")
        print("-" * 55)

    for key, meta in questions.items():
        inputs = processor(
            images=img, text=format_prompt(meta["question"]), return_tensors="pt"
        ).to(DEVICE, torch.float16)
        gen_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            num_beams=3, repetition_penalty=1.4, length_penalty=0.9,
        )
        answer = processor.tokenizer.batch_decode(
            gen_ids, skip_special_tokens=True
        )[0]
        if "Answer:" in answer:
            answer = answer.split("Answer:")[-1].strip()
        sections.append({"section": meta["section"], "answer": answer})
        raw[key] = answer
        if verbose:
            print(f"  [{meta['section']}]\n  → {answer}\n")

    date_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    lines = [
        "=" * 60,
        "       AUTOMATED RADIOLOGY REPORT (AI-ASSISTED)",
        "       *** FOR EDUCATIONAL PURPOSES ONLY ***",
        "=" * 60,
        f"Patient ID  : {patient_id}",
        f"Report Date : {date_str}",
        f"System      : Med-VQA Pipeline (BLIP-2 + LoRA)",
        "=" * 60,
    ]
    for s in sections:
        lines += [f"\n▌ {s['section'].upper()}", f"  {s['answer']}"]
    lines += [
        "\n" + "=" * 60,
        "  ⚠ AI-generated. Not for clinical use without",
        "    review by a licensed radiologist.",
        "=" * 60,
    ]
    return {
        "patient_id": patient_id, "date": date_str,
        "sections": sections, "raw_answers": raw,
        "formatted_text": "\n".join(lines),
    }

In [ ]:
# ─── Generate full report for a sample image ─────────────────────────────────
sample_idx = 10   # ← change to try different images
report = generate_report(
    image=test_ds[sample_idx]["image"],
    model=tuned_model, processor=processor,
    patient_id="PT-2024-0042", verbose=True,
)
print("\n" + report["formatted_text"])

In [ ]:
# ─── Side-by-side: image + report ────────────────────────────────────────────
fig = plt.figure(figsize=(16, 9))
gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 1.4])

ax_img = fig.add_subplot(gs[0])
ax_img.imshow(test_ds[sample_idx]["image"], cmap="gray")
ax_img.set_title("Input X-ray", fontweight="bold", fontsize=12)
ax_img.axis("off")

ax_txt = fig.add_subplot(gs[1])
ax_txt.axis("off")
display_lines = ["AUTOMATED RADIOLOGY REPORT\n", f"Patient: {report['patient_id']}  |  {report['date']}\n"]
for s in report["sections"]:
    display_lines.append(f"▌ {s['section']}")
    display_lines.append(f"  {textwrap.fill(s['answer'], 52)}\n")
ax_txt.text(0.02, 0.98, "\n".join(display_lines),
            transform=ax_txt.transAxes, va="top", fontsize=9, fontfamily="monospace",
            bbox=dict(boxstyle="round", facecolor="#f0f4f8", alpha=0.85))
fig.suptitle("Med-VQA Report Generation Pipeline — Output",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout(); plt.show()

---
## Section 8 · Interactive Demo

Upload your own X-ray or CT image and get an instant AI-generated report.

In [ ]:
import io as _io

try:
    from google.colab import files as _gfiles
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DEMO_IMAGE = None

if IN_COLAB:
    print("Upload a chest X-ray or CT image (JPG / PNG):")
    uploaded = _gfiles.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        DEMO_IMAGE = Image.open(_io.BytesIO(uploaded[fname])).convert("RGB")
        print(f"✅ Loaded: {fname}  |  size: {DEMO_IMAGE.size}")

if DEMO_IMAGE is None:
    DEMO_IMAGE = test_ds[3]["image"]
    print("Using dataset sample as demo image.")

plt.figure(figsize=(5, 5))
plt.imshow(DEMO_IMAGE, cmap="gray")
plt.title("Demo Image", fontweight="bold"); plt.axis("off"); plt.show()

In [ ]:
# ─── Single-question interactive interface ────────────────────────────────────
CUSTOM_QUESTION = "What are the main findings in this image?"  # ← Change me!

answer = predict_with_model(tuned_model, preprocess_image(DEMO_IMAGE), CUSTOM_QUESTION)
print("=" * 55)
print(f" Question : {CUSTOM_QUESTION}")
print(f" Answer   : {answer}")
print("=" * 55)

In [ ]:
# ─── Full report on demo image ────────────────────────────────────────────────
demo_report = generate_report(
    image=DEMO_IMAGE, model=tuned_model, processor=processor,
    patient_id="DEMO-0001", verbose=False,
)
print(demo_report["formatted_text"])

In [ ]:
# ─── Export report as plain text ─────────────────────────────────────────────
REPORT_OUT = "/content/generated_report.txt"
with open(REPORT_OUT, "w") as f:
    f.write(demo_report["formatted_text"])
print(f"Report saved to {REPORT_OUT}")

if IN_COLAB:
    _gfiles.download(REPORT_OUT)
    print("Downloading report...")

---
## 🎓 Summary & Key Takeaways

### What we built

```
┌──────────────────────────────────────────────────────────────────┐
│                 Med-VQA Report Generation Pipeline               │
│                                                                  │
│  X-Ray/CT  →  BLIP-2 (4-bit NF4)  →  LoRA FT (r=16)  →  Report│
│   Image       EVA-CLIP + Q-Former     OPT-2.7B adapters Generator│
└──────────────────────────────────────────────────────────────────┘
```

### Key concepts

| Concept | Implementation |
|---|---|
| **4-bit NF4 quantisation** | `BitsAndBytesConfig` → model fits in ~4 GB |
| **LoRA adapters** | Rank-16 on attention + FFN; only ~1–4M params trained |
| **Medical VQA dataset** | VQA-RAD: 3,515 clinician QA pairs + images |
| **Structured reporting** | 9-question bank → formatted clinical note |
| **Evaluation** | BLEU-1, Token F1, Exact Match, ROUGE-L |

### Recommended next steps

1. **Larger model**: `LLaVA-Med-7B` with 4-bit + LoRA for richer medical context
2. **Better data**: MIMIC-CXR radiology reports (PhysioNet credentials required)
3. **Vision encoder**: Pre-train on CheXpert / NIH ChestXray-14
4. **QLoRA**: Combine 4-bit quantisation + LoRA more tightly
5. **RAG**: Retrieve relevant clinical guidelines at inference time
6. **Clinical validation**: Radiologist review before any deployment

### ⚠️ Ethical Considerations

- This system is **not a medical device** and must not influence clinical decisions
- Validate across diverse patient populations to detect bias
- Always involve licensed radiologists when developing health AI systems

---
**References**  
- Lau et al. (2018) *Dataset and Enhancement Methods for Radiology Report Summarisation* (VQA-RAD)  
- Li et al. (2023) *BLIP-2: Bootstrapping Language-Image Pre-training with Frozen Image Encoders and LLMs*  
- Hu et al. (2022) *LoRA: Low-Rank Adaptation of Large Language Models*  
- Dettmers et al. (2023) *QLoRA: Efficient Finetuning of Quantized LLMs*

---
## Section 9 · Reflection Questions

Work through these questions to consolidate your understanding. Some are conceptual,
some require you to experiment with the notebook, and some are open-ended design challenges.

---

### 🔬 Part A — Conceptual Understanding

**A1. Quantisation trade-offs**
> When we load BLIP-2 in 4-bit NF4 quantisation, what information is lost compared to a
> full fp32 model? Why does *NormalFloat4* (NF4) tend to outperform a naive uniform 4-bit
> scheme for transformer weights? Under what circumstances might 4-bit quantisation hurt
> downstream task performance most severely?

---

**A2. Q-Former role**
> BLIP-2 inserts a Q-Former module between the frozen vision encoder and the frozen LLM.
> (a) What problem does the Q-Former solve that a simple linear projection cannot?
> (b) During zero-shot inference, which components of BLIP-2 are *actually* being updated
> (if any), and which are completely frozen?
> (c) If you removed the Q-Former and fed patch embeddings directly into OPT-2.7B, what
> failure modes would you expect?

---

**A3. LoRA mathematics**
> Given a weight matrix **W** ∈ ℝ^(4096 × 4096) and LoRA rank *r* = 16:
> (a) Calculate the number of trainable parameters introduced by one LoRA pair (A, B).
> (b) What percentage of the original matrix's parameters does this represent?
> (c) The notebook sets `lora_alpha = 32`. Explain how the effective learning-rate scaling
> factor `alpha / r` influences training stability, and what happens if you set alpha = r.

---

**A4. Label masking**
> In `MedVQADataset.__getitem__`, the prompt tokens are masked with `-100` in the labels
> tensor.
> (a) Why is this masking necessary? What would happen to the loss — and to what the model
>     learns — if we did *not* mask the prompt?
> (b) The masking uses the tokenised prompt length to set the boundary. What edge case could
>     make this boundary inaccurate, and how would you guard against it?

---

**A5. Evaluation metrics for medical NLP**
> The notebook uses BLEU-1, Token F1, Exact Match, and ROUGE-L.
> (a) For *closed* (yes/no) questions, which metric is most informative and why?
> (b) For *open-ended* findings descriptions, why might all four metrics still fall short of
>     capturing clinical quality? Name two clinically motivated metrics not used here.
> (c) A model that always outputs "yes" would achieve a non-trivial Exact Match on VQA-RAD.
>     What does this reveal about the dataset, and how should it affect how you interpret results?

---

### 🛠 Part B — Hands-On Experiments

*Modify the notebook cells to answer each question, then write down what you observe.*

**B1. LoRA rank ablation**
> Re-run Section 4 with `r ∈ {4, 8, 16, 32}`. For each rank, record:
> - Number of trainable parameters
> - Final eval loss after 3 epochs
> - Token F1 on the 80-sample eval set
>
> Plot the results. Is there a point of diminishing returns? Does higher rank always help?

---

**B2. Target-module sensitivity**
> The notebook targets `["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"]`.
> Run two additional experiments:
> - **Attention-only**: remove `fc1` and `fc2` from `TARGET_MODULES`
> - **FFN-only**: keep only `fc1` and `fc2`
>
> Compare eval loss and Token F1. Which component of the transformer benefits more from
> domain adaptation on medical VQA — the attention mechanism or the feed-forward layers?

---

**B3. Learning-rate sensitivity**
> Training LoRA with too high a learning rate can destabilise the frozen base model's
> representations. Repeat training with `learning_rate ∈ {5e-5, 2e-4, 5e-4, 1e-3}`.
> At what point do you observe divergence or degraded eval loss? What does the loss curve
> shape tell you about each setting?

---

**B4. Closed vs. open question performance**
> Filter the test set into two subsets:
> ```python
> closed = [s for s in test_ds if s["answer_type"] == "CLOSED"]
> open_  = [s for s in test_ds if s["answer_type"] == "OPEN"]
> ```
> Compute Token F1 and Exact Match separately for each subset, for both the baseline and
> fine-tuned model. Which subset benefits more from fine-tuning, and why?

---

**B5. Prompt engineering**
> The `format_prompt()` function uses a fixed template. Design and test three alternative
> prompt styles:
> 1. **Minimal**: `"Q: {question} A:"`
> 2. **Chain-of-thought**: ask the model to reason step-by-step before giving the answer
> 3. **Few-shot**: prepend two example QA pairs from the training set
>
> Without any fine-tuning, which prompt style gives the best Token F1 on 50 test samples?

---

### 🏗 Part C — System Design Challenges

**C1. Handling DICOM images**
> Real radiology workflows use DICOM files, not JPEG/PNG. The DICOM format stores 12–16 bit
> grayscale pixel arrays along with patient metadata and window/level parameters.
> (a) What preprocessing steps would you add before `preprocess_image()` to correctly
>     handle DICOM inputs? (Consider: pixel normalisation, windowing, aspect-ratio
>     preservation, and de-identification of PHI.)
> (b) Write pseudocode for a `load_dicom(path) -> PIL.Image` function using `pydicom`.

---

**C2. Multi-image report generation**
> A chest CT study often comprises 300–500 axial slices, not a single image.
> Design a pipeline that:
> 1. Selects *k* representative slices (propose a selection strategy)
> 2. Runs VQA on each selected slice
> 3. Aggregates per-slice answers into a single coherent report
>
> What are the main failure modes of naive majority-voting for answer aggregation?
> Propose a better aggregation strategy.

---

**C3. Confidence estimation**
> The current pipeline outputs a single best-beam answer with no uncertainty estimate.
> (a) Describe two approaches to derive a confidence score from the model without
>     external calibration: one based on output token probabilities, one based on
>     ensemble or sampling methods.
> (b) How would you decide when a low-confidence answer should trigger a human-in-the-loop
>     review step in a real-world deployment?

---

**C4. Catastrophic forgetting**
> LoRA is designed to reduce forgetting, but fine-tuning on a narrow domain like
> VQA-RAD could still degrade general visual-question-answering ability.
> (a) Propose an evaluation protocol to measure catastrophic forgetting using a
>     general VQA benchmark (e.g., VQAv2 or GQA).
> (b) What training strategy — other than LoRA itself — could further reduce
>     forgetting while still adapting to the medical domain?

---

**C5. Clinical deployment considerations**
> Suppose a hospital wants to use this pipeline as a *draft-report assistant* — the model
> generates an initial report that a radiologist then reviews and edits.
> Identify and discuss **four** distinct risks or failure modes that must be addressed
> before deployment, spanning: model performance, data privacy, human-AI interaction,
> and regulatory compliance.
> For each risk, propose a concrete mitigation strategy.

---

### 💡 Bonus — Going Further

> **B+1.** Replace BLIP-2/OPT-2.7B with a more recent vision-language model
> (e.g., `llava-hf/llava-1.5-7b-hf` or `google/paligemma-3b-pt-224`) loaded in 4-bit.
> Adapt the inference and fine-tuning code, then compare performance with the BLIP-2
> baseline. What architectural differences do you observe, and how do they affect
> medical VQA quality?

> **B+2.** Implement a **Retrieval-Augmented Generation (RAG)** wrapper: at inference time,
> retrieve the top-3 most similar training QA pairs using FAISS + CLIP embeddings, prepend
> them as few-shot examples in the prompt, and measure the impact on Token F1 without
> any gradient-based fine-tuning.
